# Retrieval Analysis Notebook

Offline analysis for retrieval-only devset runs already saved under `evaluator/exp/inference/devset/`.

This notebook lets you:
- reproduce evaluator-style metrics for one or more retrieval tids
- compare pairwise complementarity at top-1000
- simulate an RRF hybrid with `k=60` without re-running inference
- inspect top-20 failures with gold-track metadata


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'evaluator').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from mcrs.analysis.retrieval_analysis import (
    K_VALUES,
    build_failure_view,
    compute_pairwise_complementarity,
    evaluate_run,
    load_ground_truth,
    load_run,
    rrf_fuse_runs,
)

pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 110
print('Ready!')


## Configuration

Change the tids below to compare any saved retrieval runs. The defaults match issue #43.


In [ ]:
SPLIT = 'devset'
EXP_DIR = PROJECT_ROOT / 'evaluator' / 'exp'
TIDS = [
    'bm25_devset_retrieval_only_with_tag_list',
    'dense_qwen3_embedding_8b_devset',
]
RRF_K = 60
FAILURE_RETRIEVER = TIDS[0]  # or 'RRF'
FAILURE_SESSION_ID = None
FAILURE_TURN_NUMBER = None

CONTROL_EXPECTED = {
    'bm25_devset_retrieval_only_with_tag_list': {'ndcg@20': 0.0970, 'hit@1000': 0.6311},
    'dense_qwen3_embedding_8b_devset': {'ndcg@20': 0.1025, 'hit@1000': 0.6934},
}

print('Selected tids:')
for tid in TIDS:
    print(' -', tid)


## Load Runs, Ground Truth, and Track Metadata

In [ ]:
ground_truth = load_ground_truth(SPLIT, exp_dir=EXP_DIR)
track_metadata_ds = load_dataset(
    'talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks'
)
track_meta = pd.DataFrame(track_metadata_ds)

run_results = {}
for tid in TIDS:
    predictions = load_run(SPLIT, tid, exp_dir=EXP_DIR)
    per_instance, aggregate = evaluate_run(predictions, ground_truth)
    run_results[tid] = {
        'predictions': predictions,
        'per_instance': per_instance,
        'aggregate': aggregate,
    }

print(f'Ground-truth turns: {len(ground_truth):,}')
print(f'Track catalog size: {len(track_meta):,}')
print(f'Runs loaded: {len(run_results)}')


## Aggregate Metrics

In [ ]:
summary_rows = []
for tid, bundle in run_results.items():
    metrics = bundle['aggregate']
    summary_rows.append({
        'tid': tid,
        'ndcg@20': metrics['ndcg@20'],
        'ndcg@1000': metrics['ndcg@1000'],
        'mrr': metrics['mrr'],
        **{f'hit@{k}': metrics[f'hit@{k}'] for k in K_VALUES},
        **{f'recall@{k}': metrics[f'recall@{k}'] for k in K_VALUES},
    })

summary_df = pd.DataFrame(summary_rows).set_index('tid').sort_values('ndcg@20', ascending=False)
display(summary_df.style.format('{:.4f}'))

reproduction_rows = []
for tid, expected in CONTROL_EXPECTED.items():
    if tid not in run_results:
        continue
    actual = run_results[tid]['aggregate']
    reproduction_rows.append({
        'tid': tid,
        'expected_ndcg@20': expected['ndcg@20'],
        'actual_ndcg@20': actual['ndcg@20'],
        'expected_hit@1000': expected['hit@1000'],
        'actual_hit@1000': actual['hit@1000'],
    })

if reproduction_rows:
    reproduction_df = pd.DataFrame(reproduction_rows).set_index('tid')
    display(reproduction_df.style.format('{:.4f}'))


## Recall / Hit Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for tid, bundle in run_results.items():
    metrics = bundle['aggregate']
    hit_curve = [metrics[f'hit@{k}'] for k in K_VALUES]
    recall_curve = [metrics[f'recall@{k}'] for k in K_VALUES]
    axes[0].plot(K_VALUES, hit_curve, marker='o', label=tid)
    axes[1].plot(K_VALUES, recall_curve, marker='o', label=tid)

axes[0].set_title('Hit@K')
axes[1].set_title('Recall@K')
for ax in axes:
    ax.set_xlabel('K')
    ax.set_ylabel('Score')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Gold-Track Rank Distribution

In [ ]:
fig, axes = plt.subplots(len(run_results), 1, figsize=(10, 3.5 * len(run_results)), sharex=True)
if len(run_results) == 1:
    axes = [axes]

for ax, (tid, bundle) in zip(axes, run_results.items()):
    ranks = bundle['per_instance']['gt_rank'].dropna()
    ax.hist(ranks, bins=40, color='#4C78A8', alpha=0.85)
    ax.set_title(f'{tid} gold-track rank distribution')
    ax.set_ylabel('Turns')
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Gold-track rank when found')
plt.tight_layout()
plt.show()


## Pairwise Complementarity

The table below highlights where one retriever found the gold track in the top-1000 and the other did not.


In [ ]:
pairwise = None
if len(TIDS) < 2:
    print('Need at least two tids for pairwise complementarity.')
else:
    run_a, run_b = TIDS[:2]
    pairwise = compute_pairwise_complementarity(
        run_results[run_a]['per_instance'],
        run_results[run_b]['per_instance'],
        run_a,
        run_b,
        k=1000,
    )
    counts = (
        pairwise['complementarity']
        .value_counts()
        .rename_axis('bucket')
        .reset_index(name='n_turns')
    )
    display(counts)

    examples = pairwise[pairwise['complementarity'] != 'both_or_neither'].copy()
    display(
        examples[
            [
                'session_id',
                'turn_number',
                f'{run_a}_gt_rank',
                f'{run_b}_gt_rank',
                'complementarity',
            ]
        ].head(20)
    )


## RRF Hybrid Simulator

In [ ]:
rrf_predictions = None
rrf_instances = None
rrf_metrics = None
if len(TIDS) < 2:
    print('Need at least two tids for RRF fusion.')
else:
    rrf_predictions = rrf_fuse_runs(
        [run_results[tid]['predictions'] for tid in TIDS[:2]],
        TIDS[:2],
        rrf_k=RRF_K,
        topk=max(K_VALUES),
    )
    rrf_instances, rrf_metrics = evaluate_run(rrf_predictions, ground_truth)
    rrf_row = {
        'tid': f'RRF(k={RRF_K})',
        'ndcg@20': rrf_metrics['ndcg@20'],
        'ndcg@1000': rrf_metrics['ndcg@1000'],
        'mrr': rrf_metrics['mrr'],
        **{f'hit@{k}': rrf_metrics[f'hit@{k}'] for k in K_VALUES},
    }
    compare_df = pd.concat(
        [
            summary_df[[
                'ndcg@20', 'ndcg@1000', 'mrr', 'hit@20', 'hit@100', 'hit@200', 'hit@500', 'hit@1000'
            ]],
            pd.DataFrame([rrf_row]).set_index('tid'),
        ]
    )
    display(compare_df.style.format('{:.4f}'))


## Failure Inspection

Set `FAILURE_RETRIEVER`, `FAILURE_SESSION_ID`, and `FAILURE_TURN_NUMBER` in the config cell to inspect a specific turn.
If the session and turn are left as `None`, the notebook picks a useful example automatically.


In [ ]:
selected_predictions = None
selected_instances = None

if FAILURE_RETRIEVER == 'RRF':
    if rrf_predictions is None or rrf_instances is None:
        raise ValueError("FAILURE_RETRIEVER='RRF' requires the RRF section to run with at least two tids.")
    selected_predictions = rrf_predictions
    selected_instances = rrf_instances
else:
    selected_predictions = run_results[FAILURE_RETRIEVER]['predictions']
    selected_instances = run_results[FAILURE_RETRIEVER]['per_instance']

if FAILURE_SESSION_ID is None or FAILURE_TURN_NUMBER is None:
    if pairwise is not None and len(TIDS) >= 2:
        interesting = pairwise[pairwise['complementarity'] != 'both_or_neither']
        if not interesting.empty:
            chosen = interesting.iloc[0]
            FAILURE_SESSION_ID = chosen['session_id']
            FAILURE_TURN_NUMBER = int(chosen['turn_number'])
        else:
            chosen = selected_instances.iloc[0]
            FAILURE_SESSION_ID = chosen['session_id']
            FAILURE_TURN_NUMBER = int(chosen['turn_number'])
    else:
        chosen = selected_instances.iloc[0]
        FAILURE_SESSION_ID = chosen['session_id']
        FAILURE_TURN_NUMBER = int(chosen['turn_number'])

instance_row = selected_instances[
    (selected_instances['session_id'] == FAILURE_SESSION_ID)
    & (selected_instances['turn_number'] == FAILURE_TURN_NUMBER)
].iloc[0]

failure_view = build_failure_view(
    selected_predictions.assign(gt_rank=selected_instances['gt_rank']),
    track_meta,
    session_id=FAILURE_SESSION_ID,
    turn_number=FAILURE_TURN_NUMBER,
    gold_track_id=instance_row['gt_track_id'],
)

print(f"Retriever: {FAILURE_RETRIEVER}")
print(f"Session:   {FAILURE_SESSION_ID}")
print(f"Turn:      {FAILURE_TURN_NUMBER}")
print(f"Gold rank: {instance_row['gt_rank']}")

gold_df = pd.DataFrame([failure_view['gold_track']])
top20_df = pd.DataFrame(failure_view['top20_candidates'])
display(gold_df)
display(top20_df)
